# Higher Education Data ETL Pipeline

In [ ]:
# !pip install sqlalchemy
# !pip install psycopg2

In [ ]:
import pandas as pd
import re

# FUNCTION: Extract Year From Filename
def extract_year(file_path):
    filename = file_path.split("/")[-1]

    # Look for any 4-digit year after letters and before _ or .
    match = re.search(r"[a-zA-Z]+(\d{4})(?:_|\.|$)", filename)
    if match:
        return int(match.group(1))

    raise ValueError(f"Could not extract year from filename: {filename}")

# FUNCTION: Read & Filter a CSV in chunks
def load_filtered_chunks(
    file_path,
    usecols,
    chunksize=100_000
):
    year = extract_year(file_path)
    """
    Reads a CSV in chunks, keeps only selected columns,
    filters rows as needed, adds year of data source capture.
    """
    for chunk in pd.read_csv(
        file_path,
        usecols=usecols,
        chunksize=chunksize,
        low_memory=False
    ):
        # Normalize column names to lowercase to match postgresql
        chunk.columns = [col.lower() for col in chunk.columns]
        
        # Add year column to each chunk
        chunk["year"] = year

        # Filter rows here if needed
        yield chunk

In [ ]:
from sqlalchemy import create_engine

# FUNCTION: Upload a chunk to PostgreSQL
def upload_chunk(df, table_name, engine):
    df.to_sql(
        table_name,
        engine,
        if_exists="append",
        index=False,
        method="multi",
        chunksize=10_000
    )

In [ ]:
# FUNCTION: Process a file
def process_file(file_path, usecols, table_name, engine):
    for chunk in load_filtered_chunks(file_path, usecols):
        upload_chunk(chunk, table_name, engine)

In [ ]:
# ESTABLISH THE POSTGRESQL CONNECTION

import glob
from sqlalchemy import create_engine

# PostgreSQL connection parameters
from utility import portnumber, host, database, username, password

# Connection parameters
HOST = host
PORT = portnumber
DBNAME = database
USER = username
PASSWORD = password

# Ensure Server is running and Destination Table exists within the database
engine = create_engine(f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}")


### Integrated Postsecondary Education Data System (IPEDS)<br />

Completions - Awards/degrees conferred by program<br />
https://nces.ed.gov/ipeds/datacenter/DataFiles.aspx

In [ ]:
# PROCESS EACH FILE IN THE DIRECTORY
files = glob.glob("data/education/c20yy_a/*.csv")

for file in files:
    process_file(
        file_path=file,
        usecols= [          # Extract columns of interest to match SQL Table.
            'UNITID',
            'CIPCODE',
            'MAJORNUM',
            'AWLEVEL',
            'CTOTALT'
            ],
        table_name="edu_c20yy_a",
        engine=engine
    )

Institutional Characteristics<br />
https://nces.ed.gov/ipeds/datacenter/DataFiles.aspx

In [ ]:
# PROCESS EACH FILE IN THE DIRECTORY
files = glob.glob("data/education/hd20yy/*.csv")

for file in files:
    process_file(
        file_path=file,
        usecols= [          # Extract columns of interest to match SQL Table.
            'LONGITUD',
            'LATITUDE',
            'IALIAS',
            'ADDR',
            'CITY',
            'STABBR',
            'ZIP',
            'FIPS',
            'SECTOR',
            'ICLEVEL',
            'CONTROL',
            'HLOFFER',
            'UGOFFER',
            'GROFFER',
            'HDEGOFR1',
            'DEGGRANT',
            'COUNTYCD',
            'COUNTYNM',
            'CNGDSTCD',
            'UNITID',
            'INSTNM'
            ],
        table_name="edu_hd20yy",
        engine=engine
    )

Classification of Instructional Programs (CIP) Codes<br />
https://nces.ed.gov/ipeds/cipcode/resources.aspx?y=56

In [ ]:
# EXTRACT
edu_cipcodes_2020_df = pd.read_csv("data/education/CIPCode2020.csv")

# TRANSFORM
# Normalize column names to lowercase to match postgresql
edu_cipcodes_2020_df.columns = [col.lower() for col in edu_cipcodes_2020_df.columns]

# Remove Excel formatting
def clean_excel_format(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.startswith('="') and s.endswith('"'):
        s = s[2:-1]
    return s

for col in edu_cipcodes_2020_df.columns:
    edu_cipcodes_2020_df[col] = edu_cipcodes_2020_df[col].map(clean_excel_format)


# LOAD
DESTINATION_TABLE = "edu_cipcodes_2020"

# # Import Cleaned data to Postgresql destination table
edu_cipcodes_2020_df.to_sql(
    DESTINATION_TABLE,
    engine,
    if_exists="append", # or "replace"
    index=False,
    method="multi"      # batches inserts for speed
)